In [3]:
from pathlib import Path
import random
import subprocess

In [2]:
def make_trie_input(
    keywords_file: str | Path,
    num_keywords: int,
    seed: int | None = None,
) -> str:
    """
    Randomly selects `num_keywords` unique keywords from a line-separated file
    and returns the formatted input string for passTest2:
        <N>
        <keyword_1>
        ...
        <keyword_N>
        0
    """
    raw_lines = Path(keywords_file).read_text(encoding="utf-8").splitlines()

    # Strip whitespace, ignore blank lines, and deduplicate while preserving order
    unique_keywords = list(dict.fromkeys(
        line.strip() for line in raw_lines if line.strip()
    ))

    if num_keywords < 0:
        raise ValueError("num_keywords must be non-negative.")
    if num_keywords > len(unique_keywords):
        raise ValueError(
            f"Requested {num_keywords} unique keywords, but '{keywords_file}' "
            f"only has {len(unique_keywords)} unique non-empty lines."
        )

    rng = random.Random(seed)
    selected = rng.sample(unique_keywords, num_keywords)

    lines = [str(num_keywords), *selected, "0"]
    return "\n".join(lines) + "\n"

In [11]:

# 1. Generate the formatted input string (use seed=42 for repeatable runs)
payload = make_trie_input("Raw/INPUT.txt", num_keywords=100000, seed=128)

# 2. Feed directly into the compiled C++ executable
proc = subprocess.run(
    ["./benchFinal"],
    input=payload,
    capture_output=True,
    text=True,
    check=True,
)

# 3. Parse the 5 trie-construction lines printed by passTest2.cpp
encode_us, ac_init_us, goto_us, fail_us, stats = proc.stdout.strip().splitlines()
n_patterns, total_chars, node_count = map(int, stats.split())

print(f"Nodes: {node_count} | Encode: {encode_us} us | Init: {ac_init_us} us | Goto: {goto_us} us | Fail: {fail_us} us")
print(f"Total M: {total_chars} | Keyword Count: {n_patterns}")

ValueError: Requested 100000 unique keywords, but 'Raw/INPUT.txt' only has 10001 unique non-empty lines.

In [6]:
from pathlib import Path
import random
import subprocess
import pandas as pd

In [7]:

def run_benchmark_sweep(
    sizes: list[int] | range,
    seeds: list[int],
    keywords_file: str | Path = "Raw/INPUT.txt",
    exe_path: str | Path = "./benchFinal",
    progress_every: int = 1000,
) -> pd.DataFrame:
    # 1. Read and deduplicate keywords ONCE
    raw_lines = Path(keywords_file).read_text(encoding="utf-8").splitlines()
    unique_keywords = list(dict.fromkeys(
        line.strip() for line in raw_lines if line.strip()
    ))

    sizes_list = list(sizes)
    max_n = max(sizes_list)
    if max_n > len(unique_keywords):
        raise ValueError(f"Requested up to {max_n} keywords, but file only has {len(unique_keywords)}.")

    rows = []
    exe_str = str(exe_path)

    for seed in seeds:
        # 2. Shuffle ONCE per seed so pool[:n] grows strictly incrementally
        rng = random.Random(seed)
        pool = rng.sample(unique_keywords, max_n)

        for idx, n in enumerate(sizes_list, start=1):
            payload = f"{n}\n" + "\n".join(pool[:n]) + "\n0\n"

            proc = subprocess.run(
                [exe_str],
                input=payload,
                capture_output=True,
                text=True,
                check=True,
            )

            lines = proc.stdout.strip().splitlines()
            n_out, total_chars, node_count = map(int, lines[4].split())

            # 3. Append lightweight dicts instead of 1-row DataFrames
            rows.append({
                "seed": seed,
                "num_keywords": n_out,
                "total_chars": total_chars,
                "node_count": node_count,
                "encode_ns": int(lines[0]),
                "ac_init_ns": int(lines[1]),
                "goto_ns": int(lines[2]),
                "fail_ns": int(lines[3]),
            })

            if progress_every and idx % progress_every == 0:
                print(f"Seed {seed}: completed {idx}/{len(sizes_list)} sizes")

    # 4. Build the DataFrame ONCE
    return pd.DataFrame(rows)

In [13]:

def add_construction_time(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds a 'Construction' column equal to ac_init_ns + goto_ns + fail_ns.
    """
    return df.assign(
        Construction_us= (df["ac_init_ns"] + df["goto_ns"] + df["fail_ns"]) / 1000
    )

In [4]:
adaptive_sizes = (
    list(range(1, 101, 1))
    + list(range(110, 1001, 10))
    + list(range(1050, 10001, 50))
)
results_df_1 = run_benchmark_sweep(adaptive_sizes, seeds=[81, 24, 320, 124, 56], progress_every=100)

Seed 81: completed 100/370 sizes
Seed 81: completed 200/370 sizes
Seed 81: completed 300/370 sizes
Seed 24: completed 100/370 sizes
Seed 24: completed 200/370 sizes
Seed 24: completed 300/370 sizes
Seed 320: completed 100/370 sizes
Seed 320: completed 200/370 sizes
Seed 320: completed 300/370 sizes
Seed 124: completed 100/370 sizes
Seed 124: completed 200/370 sizes
Seed 124: completed 300/370 sizes
Seed 56: completed 100/370 sizes
Seed 56: completed 200/370 sizes
Seed 56: completed 300/370 sizes


In [14]:
results_df_1 = add_construction_time(results_df_1)
display(results_df_1)

,seed,num_keywords,total_chars,node_count,encode_ns,ac_init_ns,goto_ns,fail_ns,Construction,Construction_us
0,81,1,6,7,7400,1057,3821,2928,7.806,7.806
1,81,2,14,15,4833,1551,6822,4721,13.094,13.094
2,81,3,20,21,3955,1158,9151,4704,15.013,15.013
3,81,4,27,27,4078,1317,12768,5536,19.621,19.621
4,81,5,33,33,3702,1236,14022,7426,22.684,22.684
...,...,...,...,...,...,...,...,...,...,...
1845,56,9800,61768,26294,1099936,410158,19040297,350239113,369689.568,369689.568
1846,56,9850,62081,26407,1004062,255787,18076680,355542239,373874.706,373874.706
1847,56,9900,62386,26494,893882,264758,17726424,357941007,375932.189,375932.189
1848,56,9950,62705,26606,693531,257929,18015126,361987586,380260.641,380260.641


In [20]:
results_df_2 = (
    run_benchmark_sweep(range(1, 10001), seeds=[14, 82, 42, 128, 120])
    .pipe(add_construction_time)
)

Seed 14: completed 1000/10000 sizes
Seed 14: completed 2000/10000 sizes
Seed 14: completed 3000/10000 sizes
Seed 14: completed 4000/10000 sizes
Seed 14: completed 5000/10000 sizes
Seed 14: completed 6000/10000 sizes
Seed 14: completed 7000/10000 sizes
Seed 14: completed 8000/10000 sizes
Seed 14: completed 9000/10000 sizes
Seed 14: completed 10000/10000 sizes
Seed 82: completed 1000/10000 sizes
Seed 82: completed 2000/10000 sizes
Seed 82: completed 3000/10000 sizes
Seed 82: completed 4000/10000 sizes
Seed 82: completed 5000/10000 sizes
Seed 82: completed 6000/10000 sizes
Seed 82: completed 7000/10000 sizes
Seed 82: completed 8000/10000 sizes
Seed 82: completed 9000/10000 sizes
Seed 82: completed 10000/10000 sizes
Seed 42: completed 1000/10000 sizes
Seed 42: completed 2000/10000 sizes
Seed 42: completed 3000/10000 sizes
Seed 42: completed 4000/10000 sizes
Seed 42: completed 5000/10000 sizes
Seed 42: completed 6000/10000 sizes
Seed 42: completed 7000/10000 sizes
Seed 42: completed 8000/10

In [21]:
results_df_2

,seed,num_keywords,total_chars,node_count,encode_ns,ac_init_ns,goto_ns,fail_ns,Construction_us
0,14,1,7,8,5750,1739,4734,3232,9.705
1,14,2,11,12,4456,1238,5423,3343,10.004
2,14,3,19,20,4253,1480,11434,4632,17.546
3,14,4,23,24,4037,1150,11776,4783,17.709
4,14,5,28,29,4422,2115,13445,5724,21.284
...,...,...,...,...,...,...,...,...,...
49995,120,9996,62998,26713,1212139,284150,18520666,359430123,378234.939
49996,120,9997,63002,26714,724327,261428,18361199,361237608,379860.235
49997,120,9998,63007,26714,689660,255185,17454950,367155379,384865.514
49998,120,9999,63015,26718,1112141,418059,21005580,361206066,382629.705


In [22]:
results_df_2.to_csv("results_2.csv", index=False)